Shrushti Sakat      
INTERNSHIP ID : IN226049402

In [2]:
# Install required libraries (run once)
!pip install nltk --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import nltk
from collections import Counter

# Download necessary NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---
##  Task 1: Conceptual Understanding (Written Answers)

### Q1. What is the difference between `"Love"` and `"love"` in NLP?

In NLP, `"Love"` and `"love"` are treated as **different tokens** by default because most systems are case-sensitive. However, they carry the same semantic meaning. This is why **lowercasing** is a fundamental preprocessing step — it ensures that the same word in different capitalizations is treated as one unified token, improving model consistency and reducing vocabulary size.

---

### Q2. What happens if stopwords are not removed?

If stopwords (e.g., *is, the, a, and, of*) are not removed:
- They dominate **frequency-based analyses** and distort results.
- ML models give **unnecessary weight** to these low-information words.
- **Vocabulary size** increases, raising computational cost.
- The signal-to-noise ratio in feature vectors decreases, reducing model accuracy.

---

### Q3. Two real-world scenarios where removing stopwords can be HARMFUL:

1. **Sentiment Analysis involving negations:**  
   The sentence `"I am NOT happy"` becomes `"happy"` after removing stopwords — which completely reverses the sentiment. The word *"not"* is critical here.

2. **Question Answering / Information Retrieval:**  
   In a query like `"Who is the president of France?"`, removing stopwords gives `"president France"` — which loses the intent and structure of the question. Systems relying on exact phrase matching will fail.

---

### Q4. Difference between Stemming and Lemmatization:

| Feature | Stemming | Lemmatization |
|---|---|---|
| **Method** | Chops word endings using heuristic rules | Reduces word to its dictionary base form |
| **Output** | May not be a real word | Always a valid word |
| **Example** | `"running"` → `"run"`, `"studies"` → `"studi"` | `"running"` → `"run"`, `"studies"` → `"study"` |
| **Speed** | Faster | Slower (uses vocabulary + grammar) |
| **Accuracy** | Lower | Higher |
| **Use Case** | Quick, large-scale search indexing | High-accuracy NLP models |


---
##  Task 2: Build Advanced Preprocessing Function

In [4]:
# Initialize NLP tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Words that are short but meaningful — should NOT be removed
MEANINGFUL_SHORT_WORDS = {"no", "not", "ok", "go"}


def preprocess_text(text):
    """
    Advanced NLP Preprocessing Function.
    
    Steps:
      1. Validate input
      2. Remove URLs and email-like patterns
      3. Remove emojis and non-ASCII characters
      4. Convert to lowercase
      5. Handle repeated characters (e.g., 'soooo' → 'so')
      6. Remove punctuation and special characters
      7. Remove numbers
      8. Tokenize
      9. Remove extra short tokens (len ≤ 2), keeping meaningful exceptions
     10. Lemmatize tokens
     11. Remove extra whitespace
    
    Args:
        text (str): Raw input text
    
    Returns:
        tuple: (list of cleaned tokens, cleaned sentence string)
    """
    
    # --- Step 1: Input Validation ---
    if not isinstance(text, str) or not text.strip():
        return [], ""
    
    # --- Step 2: Remove URLs and email-like patterns ---
    # Matches http/https URLs and www patterns
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    
    # --- Step 3: Remove emojis and non-ASCII characters ---
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # --- Step 4: Convert to lowercase ---
    text = text.lower()
    
    # --- Step 5: Handle repeated characters ---
    # Reduce any character repeated 3+ times to just 1 occurrence
    # e.g., 'soooo' → 'so', 'baaad' → 'bad', '!!!' → '!'
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # --- Step 6: Remove punctuation and special characters ---
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # --- Step 7: Remove numbers (already handled above, but explicit) ---
    text = re.sub(r'\b\d+\b', '', text)
    
    # --- Step 8: Tokenize ---
    tokens = text.split()
    
    # --- Step 9: Remove very short tokens, keep meaningful exceptions ---
    tokens = [
        token for token in tokens
        if len(token) > 2 or token in MEANINGFUL_SHORT_WORDS
    ]
    
    # --- Step 10: Lemmatize tokens ---
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # --- Step 11: Final cleaned sentence ---
    clean_sentence = ' '.join(tokens)
    
    return tokens, clean_sentence


print("✅ preprocess_text() function defined successfully!")

# Quick sanity checks
print("\n--- Quick Sanity Checks ---")
print(preprocess_text("I have 2 dogs"))               # Remove numbers
print(preprocess_text("This  is   good"))              # Remove extra spaces
print(preprocess_text("soooo goooood!!!"))             # Repeated chars
print(preprocess_text("WOW!!! This is GREAT!!!"))      # Lowercase
print(preprocess_text("Visit http://example.com now")) # URLs
print(preprocess_text("I am not happy with this"))     # Keep 'not'

✅ preprocess_text() function defined successfully!

--- Quick Sanity Checks ---
(['have', 'dog'], 'have dog')
(['this', 'good'], 'this good')
(['god'], 'god')
(['wow', 'this', 'great'], 'wow this great')
(['visit', 'now'], 'visit now')
(['not', 'happy', 'with', 'this'], 'not happy with this')


---
##  Task 3: Stress Testing — 10 Diverse Sentences

In [5]:
# Sample input sentences for stress testing
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("=" * 70)
print("STRESS TESTING RESULTS")
print("=" * 70)

results = []  # Store results for later tasks

for i, sentence in enumerate(sample_inputs, 1):
    tokens, clean_sentence = preprocess_text(sentence)
    results.append({
        "original": sentence,
        "tokens": tokens,
        "clean_sentence": clean_sentence
    })
    
    print(f"\n[{i}] Original Text    : {sentence}")
    print(f"    Cleaned Tokens   : {tokens}")
    print(f"    Cleaned Sentence : {clean_sentence}")
    print("-" * 70)

STRESS TESTING RESULTS

[1] Original Text    : Get 100% FREE access now!!!
    Cleaned Tokens   : ['get', 'free', 'access', 'now']
    Cleaned Sentence : get free access now
----------------------------------------------------------------------

[2] Original Text    : I absolutely looooved this product 😍😍
    Cleaned Tokens   : ['absolutely', 'loved', 'this', 'product']
    Cleaned Sentence : absolutely loved this product
----------------------------------------------------------------------

[3] Original Text    : Worst service ever... 0/10
    Cleaned Tokens   : ['worst', 'service', 'ever']
    Cleaned Sentence : worst service ever
----------------------------------------------------------------------

[4] Original Text    : Call me at 9876543210
    Cleaned Tokens   : ['call']
    Cleaned Sentence : call
----------------------------------------------------------------------

[5] Original Text    : This is THE best course!!!
    Cleaned Tokens   : ['this', 'the', 'best', 'course']
  

---
##  Task 4: Token Analytics

In [6]:
print("=" * 90)
print(f"{'#':<4} {'Original (truncated)':<35} {'Total':>6} {'Unique':>7} {'Avg Len':>8}")
print("=" * 90)

for i, res in enumerate(results, 1):
    tokens = res['tokens']
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_len = round(sum(len(t) for t in tokens) / total_tokens, 2) if tokens else 0
    
    # Store analytics back for reference
    res['total_tokens'] = total_tokens
    res['unique_tokens'] = unique_tokens
    res['avg_len'] = avg_len
    
    truncated = res['original'][:33] + ('...' if len(res['original']) > 33 else '')
    print(f"{i:<4} {truncated:<35} {total_tokens:>6} {unique_tokens:>7} {avg_len:>8}")

print("=" * 90)

# --- Analysis Questions ---
print("\n📌 Analysis Questions:")

# Sentence with most noise = largest drop from original word count to clean token count
noise_scores = []
for res in results:
    original_word_count = len(res['original'].split())
    remaining = res['total_tokens']
    # Noise = proportion of words lost
    noise = original_word_count - remaining
    noise_scores.append(noise)

max_noise_idx = noise_scores.index(max(noise_scores))
print(f"\n🔴 Most Noisy Sentence (#{max_noise_idx+1}):")
print(f"   '{results[max_noise_idx]['original']}'")
print(f"   Lost {noise_scores[max_noise_idx]} word(s) during cleaning.")

# Sentence retaining most meaningful tokens
max_retain_idx = max(range(len(results)), key=lambda i: results[i]['total_tokens'])
print(f"\n🟢 Most Meaningful Tokens Retained (#{max_retain_idx+1}):")
print(f"   '{results[max_retain_idx]['original']}'")
print(f"   Retained {results[max_retain_idx]['total_tokens']} token(s): {results[max_retain_idx]['tokens']}")

#    Original (truncated)                 Total  Unique  Avg Len
1    Get 100% FREE access now!!!              4       4      4.0
2    I absolutely looooved this produc...      4       4      6.5
3    Worst service ever... 0/10               3       3     5.33
4    Call me at 9876543210                    1       1      4.0
5    This is THE best course!!!               4       4     4.25
6    Visit https://openai.com now!            2       2      4.0
7    Nooooo this is baaad!!!                  3       3      3.0
8    OK OK OK I got it                        4       2     2.25
9    Win $$$ now!!! Limited offer!!!          4       4      4.5
10   I am not happy with this                 4       4      4.0

📌 Analysis Questions:

🔴 Most Noisy Sentence (#4):
   'Call me at 9876543210'
   Lost 3 word(s) during cleaning.

🟢 Most Meaningful Tokens Retained (#1):
   'Get 100% FREE access now!!!'
   Retained 4 token(s): ['get', 'free', 'access', 'now']


---
##  Task 5: Frequency Analysis

In [7]:
from collections import Counter

# Combine ALL tokens from all processed sentences
all_tokens = []
for res in results:
    all_tokens.extend(res['tokens'])

print(f"Total tokens across all sentences: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}\n")

# Compute frequency
token_freq = Counter(all_tokens)

# Top 10 most frequent words
print("🔝 Top 10 Most Frequent Words:")
print("-" * 30)
for rank, (word, count) in enumerate(token_freq.most_common(10), 1):
    print(f"  {rank:>2}. '{word}' → {count} time(s)")

# Top 5 least frequent words
print("\n🔻 Top 5 Least Frequent Words:")
print("-" * 30)
least_common = token_freq.most_common()[:-6:-1]  # Last 5
for rank, (word, count) in enumerate(least_common, 1):
    print(f"  {rank:>2}. '{word}' → {count} time(s)")

Total tokens across all sentences: 33
Unique tokens: 26

🔝 Top 10 Most Frequent Words:
------------------------------
   1. 'this' → 4 time(s)
   2. 'now' → 3 time(s)
   3. 'ok' → 3 time(s)
   4. 'get' → 1 time(s)
   5. 'free' → 1 time(s)
   6. 'access' → 1 time(s)
   7. 'absolutely' → 1 time(s)
   8. 'loved' → 1 time(s)
   9. 'product' → 1 time(s)
  10. 'worst' → 1 time(s)

🔻 Top 5 Least Frequent Words:
------------------------------
   1. 'with' → 1 time(s)
   2. 'happy' → 1 time(s)
   3. 'not' → 1 time(s)
   4. 'offer' → 1 time(s)
   5. 'limited' → 1 time(s)


---
##  Task 6: Build Full Pipeline

In [8]:
def full_pipeline(text_list):
    """
    Full NLP Preprocessing Pipeline.
    
    Accepts a list of raw text strings and returns a dictionary
    containing all cleaned tokens and cleaned sentences.
    
    Args:
        text_list (list): List of raw text strings.
    
    Returns:
        dict: {
            'tokens': list of token lists per sentence,
            'clean_sentences': list of cleaned sentence strings
        }
    """
    
    # Validate input
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list of strings.")
    
    all_tokens = []
    clean_sentences = []
    
    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        all_tokens.append(tokens)
        clean_sentences.append(clean_sentence)
    
    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }


print("✅ full_pipeline() function defined!")

# Run full pipeline on the sample inputs
pipeline_output = full_pipeline(sample_inputs)

print("\n📤 Full Pipeline Output:")
print("=" * 60)

for i, (tokens, sentence) in enumerate(
    zip(pipeline_output['tokens'], pipeline_output['clean_sentences']), 1
):
    print(f"[{i}] Tokens  : {tokens}")
    print(f"    Sentence: {sentence}")
    print()

✅ full_pipeline() function defined!

📤 Full Pipeline Output:
[1] Tokens  : ['get', 'free', 'access', 'now']
    Sentence: get free access now

[2] Tokens  : ['absolutely', 'loved', 'this', 'product']
    Sentence: absolutely loved this product

[3] Tokens  : ['worst', 'service', 'ever']
    Sentence: worst service ever

[4] Tokens  : ['call']
    Sentence: call

[5] Tokens  : ['this', 'the', 'best', 'course']
    Sentence: this the best course

[6] Tokens  : ['visit', 'now']
    Sentence: visit now

[7] Tokens  : ['no', 'this', 'bad']
    Sentence: no this bad

[8] Tokens  : ['ok', 'ok', 'ok', 'got']
    Sentence: ok ok ok got

[9] Tokens  : ['win', 'now', 'limited', 'offer']
    Sentence: win now limited offer

[10] Tokens  : ['not', 'happy', 'with', 'this']
    Sentence: not happy with this



---
##  Task 7: Error Handling — Edge Cases

In [9]:
# Define edge case inputs
edge_cases = [
    "",                   # Empty string
    "   ",                # Whitespace only
    "😍😍🔥💯🎉",        # Only emojis
    "1234567890",         # Only numbers
    "!!! $$$ ###",        # Only special characters
    None,                 # None type (defensive handling)
    123,                  # Non-string type (defensive handling)
]

print("=" * 60)
print("EDGE CASE / ERROR HANDLING TESTS")
print("=" * 60)

for case in edge_cases:
    try:
        tokens, sentence = preprocess_text(case)
        print(f"Input  : {repr(case)}")
        print(f"Tokens : {tokens}")
        print(f"Output : '{sentence}'")
    except Exception as e:
        print(f"Input  : {repr(case)}")
        print(f"⚠️  Error caught gracefully: {type(e).__name__}: {e}")
    print("-" * 60)

EDGE CASE / ERROR HANDLING TESTS
Input  : ''
Tokens : []
Output : ''
------------------------------------------------------------
Input  : '   '
Tokens : []
Output : ''
------------------------------------------------------------
Input  : '😍😍🔥💯🎉'
Tokens : []
Output : ''
------------------------------------------------------------
Input  : '1234567890'
Tokens : []
Output : ''
------------------------------------------------------------
Input  : '!!! $$$ ###'
Tokens : []
Output : ''
------------------------------------------------------------
Input  : None
Tokens : []
Output : ''
------------------------------------------------------------
Input  : 123
Tokens : []
Output : ''
------------------------------------------------------------
